# DALE vs STREME: Live Head-to-Head Benchmark

**Both tools run live. No pre-computed results.**

DALE is a 906 KB static binary (zero dependencies). STREME comes from the MEME Suite.
Both receive the same ENCODE K562 ChIP-seq TFs, discover motifs independently,
and are scored with identical AUROC evaluation on dinucleotide-shuffled negatives.

**Click Runtime → Run All. Total time: ~30 seconds.**

Paper: [doi.org/10.5281/zenodo.21907349](https://doi.org/10.5281/zenodo.21907349)


## 1. Install MEME Suite & Download DALE


In [ ]:
import os, subprocess, sys, shutil, time

MEME_VERSION = '5.5.9'
MEME_TARBALL = f'meme-{MEME_VERSION}.tar.gz'
MEME_URL = f'https://meme-suite.org/meme-software/{MEME_VERSION}/{MEME_TARBALL}'
MEME_DIR = '/opt/meme'

# --- Install MEME Suite from source ---
if not os.path.exists(f'{MEME_DIR}/bin/streme'):
    # Install system XML dev libraries (required by MEME, avoids build failures)
    subprocess.run('apt-get install -y libxml2-dev libxslt1-dev 2>&1 | tail -3',
                   shell=True, executable='/bin/bash', capture_output=True)
    
    print(f'Downloading MEME Suite {MEME_VERSION}...')
    subprocess.run(f'wget -q {MEME_URL} -O /tmp/{MEME_TARBALL}', shell=True, check=True)
    
    print('Extracting...')
    subprocess.run(f'cd /tmp && tar xzf {MEME_TARBALL}', shell=True, check=True)
    
    src_dir = f'/tmp/meme-{MEME_VERSION}'
    print(f'Configuring (using system XML libraries)...')
    r = subprocess.run(
        f'cd {src_dir} && ./configure --prefix={MEME_DIR} --enable-build-libraries=no 2>&1 | tail -10',
        shell=True, capture_output=True, text=True, timeout=300
    )
    print(r.stdout[-600:] if r.stdout else '(no output)')
    
    print('Building (make)...')
    r = subprocess.run(
        f'cd {src_dir} && make -j$(nproc) 2>&1 | tail -10',
        shell=True, capture_output=True, text=True, timeout=600
    )
    print(r.stdout[-600:] if r.stdout else '(no output)')
    
    print('Installing...')
    r = subprocess.run(
        f'cd {src_dir} && make install 2>&1 | tail -5',
        shell=True, capture_output=True, text=True, timeout=300
    )
    print(r.stdout[-400:] if r.stdout else '(no output)')
    print(f'Done. MEME Suite installed to {MEME_DIR}')

streme_ok = os.path.exists(f'{MEME_DIR}/bin/streme')
meme_ok = os.path.exists(f'{MEME_DIR}/bin/meme')
os.environ['PATH'] = f'{MEME_DIR}/bin:{MEME_DIR}/libexec/meme-{MEME_VERSION}:{os.environ.get("PATH", "")}'

# --- Download DALE ---
if not os.path.isdir('little-scientist-dale'):
    subprocess.run(
        'rm -rf little-scientist-dale && git clone -q '
        'https://github.com/Travis42/little-scientist-dale.git little-scientist-dale',
        shell=True, check=True
    )
os.chdir('little-scientist-dale')

DALE_BIN = './bin/dale'
if not os.access(DALE_BIN, os.X_OK):
    os.chmod(DALE_BIN, 0o755)

n_tfs = len([f for f in os.listdir('example') if f.endswith('.fa')])

print(f'\nDALE:   {os.path.getsize(DALE_BIN)//1024} KB (static binary, zero deps)')
print(f'{"OK" if streme_ok else "MISSING"} STREME: {MEME_DIR}/bin/streme')
print(f'{"OK" if meme_ok else "MISSING"} MEME:   {MEME_DIR}/bin/meme')
print(f'Data:   {n_tfs} TFs from ENCODE K562 ChIP-seq in example/')

if not streme_ok:
    raise RuntimeError('STREME not found. MEME Suite installation failed.')


## 2. Compile Benchmark Binary

We compile from source (C99 + libm only). The binary runs DALE then STREME
on each TF and scores both with identical AUROC evaluation.
The `--no-meme` flag skips the MEME comparison for faster execution.

In [ ]:
import os, subprocess, shutil

os.chdir('src')

# Fix license header for gcc compatibility
with open('dale.c', 'r') as f:
    lines = f.readlines()
end = 0
for i, line in enumerate(lines):
    if line.startswith('# ') and i < 20:
        end = i + 1
    elif end > 0 and not line.startswith('#'):
        break
if end > 0:
    new_lines = ['/*\n']
    for line in lines[:end]:
        new_lines.append(' * ' + line.lstrip('# ').rstrip('\n') + '\n')
    new_lines.append(' */\n\n')
    new_lines.extend(lines[end:])
    with open('dale.c', 'w') as f:
        f.writelines(new_lines)

# Compile
subprocess.run('cc -std=c99 -O2 -Wall -Wextra -Wno-unused-parameter -c dale.c -o dale.o',
               shell=True, check=True)
subprocess.run('cc -std=c99 -O2 -Wall -c benchmark.c -o benchmark.o',
               shell=True, check=True)
subprocess.run('cc -o benchmark dale.o benchmark.o -lm', shell=True, check=True)
shutil.copy('benchmark', '../benchmark')
os.chdir('..')
print(f'Benchmark binary: {os.path.getsize("benchmark")} bytes')



## 2. Run DALE vs STREME (Live Head-to-Head)

The benchmark binary runs DALE then STREME on each TF, scoring both with
identical AUROC evaluation. MEME is excluded (legacy tool; results available
in pre-computed CSV files in the repo).

*(~30 seconds for 11 TFs.)*

In [ ]:
import subprocess, time

t0 = time.time()
r = subprocess.run(
    ['./benchmark', '--data', 'example/', '--tf', 'ALL', '--no-meme'],
    capture_output=True, text=True, timeout=120
)
elapsed = time.time() - t0
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[:300])
print(f'\nWall time: {elapsed:.1f}s')

## 3. Results & Visualization


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wilcoxon

# Parse benchmark output
rows = []
for line in r.stdout.strip().split('\n'):
    parts = line.split('\t')
    if len(parts) >= 5 and parts[0] != 'TF':
        try:
            rows.append({
                'TF': parts[0], 'Width': int(parts[1]),
                'AUROC': float(parts[2]), 'Time_s': float(parts[3]),
                'Source': parts[4]
            })
        except (ValueError, IndexError):
            pass

df = pd.DataFrame(rows)
dale = df[df['Source'] == 'ours'].rename(columns={'AUROC': 'DALE_AUROC', 'Time_s': 'DALE_Time'})
streme = df[df['Source'] == 'streme'].rename(columns={'AUROC': 'STREME_AUROC', 'Time_s': 'STREME_Time'})

merged = dale[['TF', 'DALE_AUROC', 'DALE_Time']].merge(
    streme[['TF', 'STREME_AUROC', 'STREME_Time']], on='TF'
)

print(f'Tools compared on {len(merged)} TFs')
print()

MD_COLOR = '#2166AC'
ST_COLOR = '#D6604D'

print(f'DALE:   avg AUROC = {merged["DALE_AUROC"].mean():.4f}  avg time = {merged["DALE_Time"].mean():.2f}s/TF')
print(f'STREME: avg AUROC = {merged["STREME_AUROC"].mean():.4f}  avg time = {merged["STREME_Time"].mean():.2f}s/TF')
print()

d_vs_s = merged['DALE_AUROC'].values - merged['STREME_AUROC'].values
_, p = wilcoxon(d_vs_s)
wins_s = (d_vs_s > 0.001).sum()
loss_s = (d_vs_s < -0.001).sum()
ties = len(d_vs_s) - wins_s - loss_s
print(f'DALE vs STREME: {d_vs_s.mean():+.4f}, Wilcoxon p = {p:.2e} ({wins_s}W/{ties}T/{loss_s}L)')

# Figure: scatter + speed
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: DALE vs STREME scatter
o = merged['DALE_AUROC'].values
s = merged['STREME_AUROC'].values
colors = np.where(o > s, MD_COLOR, ST_COLOR)
ax1.scatter(s, o, alpha=0.7, s=50, c=colors, edgecolors='black', linewidths=0.5)
ax1.plot([0.4, 1.0], [0.4, 1.0], 'k--', alpha=0.3)
ax1.set_xlabel('STREME AUROC', fontsize=11)
ax1.set_ylabel('DALE AUROC', fontsize=11)
ax1.set_title(f'DALE vs STREME ({wins_s}W/{loss_s}L)', fontsize=12, fontweight='bold')
for _, row in merged.iterrows():
    d = row['DALE_AUROC'] - row['STREME_AUROC']
    if abs(d) > 0.12:
        ax1.annotate(row['TF'], (row['STREME_AUROC'], row['DALE_AUROC']), fontsize=8)

# Panel B: Speed comparison
tools = ['DALE', 'STREME']
times = [merged['DALE_Time'].mean(), merged['STREME_Time'].mean()]
colors = [MD_COLOR, ST_COLOR]
bars = ax2.bar(tools, times, color=colors, edgecolor='black', width=0.4)
for bar, t in zip(bars, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.03,
            f'{t:.1f}s/TF', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylabel('Time per TF (seconds)', fontsize=11)
ax2.set_title('Speed Comparison', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('head_to_head_results.png', dpi=150, bbox_inches='tight')
plt.show()



## 4. Per-TF Details


In [ ]:
display_cols = ['TF', 'DALE_AUROC', 'STREME_AUROC', 'DALE_Time', 'STREME_Time']
detail = merged[display_cols].copy()
detail['DALE_wins'] = detail['DALE_AUROC'] > detail['STREME_AUROC']
detail = detail.sort_values('DALE_AUROC', ascending=False)
print(detail.to_string(index=False))
print(f'\nDALE wins on {detail["DALE_wins"].sum()} of {len(detail)} TFs')



---

**Reproducibility checklist:**
- All three tools run from source with default parameters
- Same input sequences (11 ENCODE K562 ChIP-seq TFs)
- Same AUROC metric on dinucleotide-shuffled negatives (seed=42)
- No pre-computed results — everything runs live
- DALE binary is static, zero dependencies; STREME built from MEME Suite source
- Full paper + code: [github.com/Travis42/little-scientist-dale](https://github.com/Travis42/little-scientist-dale)
- MEME comparison available in pre-computed 132-TF CSV: `shuffled_negatives_132tf.csv`

